# Pull Clean and Aggregate data by local site, county, and date

## PM2.5

In [36]:
import pandas as pd

# ── 1. LOAD & FILTER ──────────────────────────────────────────────────────────
# Only pull the columns we actually need. State Name is required for the NJ
# filter — do not remove it.

cols_needed = [
    'State Name', 'Date Local', 'County Name', 'Local Site Name',
    'Latitude', 'Longitude', 'Arithmetic Mean', 'AQI',
    '1st Max Value', 'Sample Duration'
]

pm25_raw = pd.concat([
    pd.read_csv('data/PM2.5/PM2.5_daily_88101_2023.csv', low_memory=False, usecols=cols_needed),
    pd.read_csv('data/PM2.5/PM2.5_daily_88101_2024.csv', low_memory=False, usecols=cols_needed),
    pd.read_csv('data/PM2.5/PM2.5_daily_88101_2025.csv', low_memory=False, usecols=cols_needed),
], ignore_index=True)

# Filter to New Jersey only
pm25_nj = pm25_raw[pm25_raw['State Name'] == 'New Jersey'].copy()

print(f"Total NJ rows (all durations): {len(pm25_nj)}")

# ── 2. FILTER TO 24-HOUR BLOCK AVERAGE ───────────────────────────────────────
# PM2.5 AQI is officially derived from the 24-hour block average — one reading
# per station per calendar day. Unlike NO2 (1-hour) and Ozone (8-hour rolling),
# this filter already gives us one row per station per day, so no daily
# collapse step is needed.

pm25_24hr = pm25_nj[pm25_nj['Sample Duration'] == '24-HR BLK AVG'].copy()

print(f"NJ rows after 24-hour filter: {len(pm25_24hr)}")

# ── 3. PARSE DATES ────────────────────────────────────────────────────────────
pm25_24hr['Date Local'] = pd.to_datetime(pm25_24hr['Date Local'], format='mixed')
pm25_24hr['Year'] = pm25_24hr['Date Local'].dt.year

# ── 4. SAVE INTERMEDIATE DAILY FILE ──────────────────────────────────────────
# One row per station per calendar day — already guaranteed by the 24-HR BLK
# AVG filter. This is the file to use for Phase 2 NFL game-day comparisons
# and summer-only slicing (June–July).

pm25_24hr.to_csv('data/PM2.5/PM2.5_daily_clean_2023-2025.csv', index=False)
print("Saved: data/PM2.5/PM2.5_daily_clean_2023-2025.csv")

# ── 5. AGGREGATE TO ANNUAL SUMMARY ───────────────────────────────────────────
# One row per station per year.
#
#   avg_reading   — mean of daily 24-hr averages (true annual baseline)
#   avg_aqi       — mean of daily AQI values — correct here because 24-HR BLK
#                   AVG is already the basis for the official PM2.5 AQI
#   peak_reading  — single worst day across the whole year
#   days_recorded — nunique on Date Local (explicit calendar day count)
#                   NOTE: with 24-HR BLK AVG this will match count(), but
#                   nunique is the correct and explicit way to express intent

pm25_summary = pm25_24hr.groupby(
    ['Local Site Name', 'County Name', 'Latitude', 'Longitude', 'Year']
).agg(
    avg_reading   = ('Arithmetic Mean', 'mean'),
    avg_aqi       = ('AQI', 'mean'),
    peak_reading  = ('1st Max Value', 'max'),
    days_recorded = ('Date Local', 'nunique')   # nunique = explicit calendar days
).reset_index()

# ── 6. SAVE ANNUAL SUMMARY ────────────────────────────────────────────────────
pm25_summary.to_csv('data/PM2.5/PM2.5_summary_2023-2025.csv', index=False)
print("Saved: data/PM2.5/PM2.5_summary_2023-2025.csv")
print(pm25_summary.to_string())

Total NJ rows (all durations): 34353
NJ rows after 24-hour filter: 15420
Saved: data/PM2.5/PM2.5_daily_clean_2023-2025.csv
Saved: data/PM2.5/PM2.5_summary_2023-2025.csv
           Local Site Name County Name   Latitude  Longitude  Year  avg_reading    avg_aqi  peak_reading  days_recorded
0            Atlantic City    Atlantic  39.363260 -74.431000  2023     7.133702  36.187845          24.4            181
1            Atlantic City    Atlantic  39.363260 -74.431000  2024     5.497207  28.907821          19.9            358
2            Atlantic City    Atlantic  39.363260 -74.431000  2025     5.693452  29.553571          21.7            168
3               Brigantine    Atlantic  39.464872 -74.448736  2023     8.232081  38.040462         139.8            346
4               Brigantine    Atlantic  39.464872 -74.448736  2024     5.559416  28.974026          33.9            308
5               Brigantine    Atlantic  39.464872 -74.448736  2025     5.145783  27.132530          18.8       

## Ozone

In [ ]:
import pandas as pd

# ── 1. LOAD & FILTER ──────────────────────────────────────────────────────────
# Only pull the columns we actually need. State Name is required for the NJ
# filter — do not remove it.

cols_needed = [
    'State Name', 'Date Local', 'County Name', 'Local Site Name',
    'Latitude', 'Longitude', 'Arithmetic Mean', 'AQI',
    '1st Max Value', 'Sample Duration'
]

ozone_raw = pd.concat([
    pd.read_csv('data/Ozone/Ozone_daily_44201_2023.csv', low_memory=False, usecols=cols_needed),
    pd.read_csv('data/Ozone/Ozone_daily_44201_2024.csv', low_memory=False, usecols=cols_needed),
    pd.read_csv('data/Ozone/Ozone_daily_44201_2025.csv', low_memory=False, usecols=cols_needed),
], ignore_index=True)

# Filter to New Jersey only
ozone_nj = ozone_raw[ozone_raw['State Name'] == 'New Jersey'].copy()

print(f"Total NJ rows (all durations): {len(ozone_nj)}")

# ── 2. FILTER TO 8-HOUR READINGS ──────────────────────────────────────────────
# Ozone AQI is officially derived from the daily maximum 8-hour average.
# The EPA uses the highest 8-hour rolling average of the day as the basis
# for the AQI calculation — not a 24-hour mean.

ozone_8hr = ozone_nj[ozone_nj['Sample Duration'] == '8-HR RUN AVG BEGIN HOUR'].copy()

print(f"NJ rows after 8-hour filter: {len(ozone_8hr)}")

# ── 3. PARSE DATES ────────────────────────────────────────────────────────────
ozone_8hr['Date Local'] = pd.to_datetime(ozone_8hr['Date Local'], format='mixed')
ozone_8hr['Year'] = ozone_8hr['Date Local'].dt.year

# ── 4. COLLAPSE TO ONE ROW PER STATION PER CALENDAR DAY ──────────────────────
# Each calendar day can have multiple overlapping 8-hour windows per station.
# Before aggregating to the annual summary, reduce to one row per station per
# day.
#
# Per-day values:
#   daily_mean_reading  — average of all 8-hr readings that day
#   daily_max_aqi       — highest AQI of the day (matches EPA ozone AQI method)
#   daily_1st_max       — the 1st Max Value field (EPA-computed daily max)

ozone_daily = ozone_8hr.groupby(
    ['Local Site Name', 'County Name', 'Latitude', 'Longitude', 'Year', 'Date Local']
).agg(
    daily_mean_reading = ('Arithmetic Mean', 'mean'),
    daily_max_aqi      = ('AQI', 'max'),        # official ozone AQI uses daily 8-hr max
    daily_1st_max      = ('1st Max Value', 'max')
).reset_index()

print(f"Rows after daily collapse: {len(ozone_daily)}")

# ── 5. SAVE INTERMEDIATE DAILY FILE ──────────────────────────────────────────
# One row per station per calendar day — the correct grain for Phase 2
# NFL game-day comparisons and summer-only slicing (June–July).

ozone_daily.to_csv('data/Ozone/Ozone_daily_clean_2023-2025.csv', index=False)
print("Saved: data/Ozone/Ozone_daily_clean_2023-2025.csv")

# ── 6. AGGREGATE TO ANNUAL SUMMARY ───────────────────────────────────────────
# One row per station per year.
#
#   avg_reading   — mean of daily averages (annual baseline)
#   avg_aqi       — mean of daily MAX AQI values (matches EPA reporting method)
#   peak_reading  — single worst day across the whole year
#   days_recorded — count of unique calendar days (nunique prevents double-count)

ozone_summary = ozone_daily.groupby(
    ['Local Site Name', 'County Name', 'Latitude', 'Longitude', 'Year']
).agg(
    avg_reading   = ('daily_mean_reading', 'mean'),
    avg_aqi       = ('daily_max_aqi', 'mean'),      # mean of daily maximums
    peak_reading  = ('daily_1st_max', 'max'),
    days_recorded = ('Date Local', 'nunique')
).reset_index()

# ── 7. SAVE ANNUAL SUMMARY ────────────────────────────────────────────────────
ozone_summary.to_csv('data/Ozone/Ozone_summary_2023-2025.csv', index=False)
print("Saved: data/Ozone/Ozone_summary_2023-2025.csv")
print(ozone_summary.to_string())


## NO2

In [ ]:
import pandas as pd

# ── 1. LOAD & FILTER ──────────────────────────────────────────────────────────
# Only pull the columns we actually need. State Name is required for the NJ
# filter — do not remove it.

cols_needed = [
    'State Name', 'Date Local', 'County Name', 'Local Site Name',
    'Latitude', 'Longitude', 'Arithmetic Mean', 'AQI',
    '1st Max Value', 'Sample Duration'
]

no2_raw = pd.concat([
    pd.read_csv('data/NO2/NO2_daily_42602_2023.csv', low_memory=False, usecols=cols_needed),
    pd.read_csv('data/NO2/NO2_daily_42602_2024.csv', low_memory=False, usecols=cols_needed),
    pd.read_csv('data/NO2/NO2_daily_42602_2025.csv', low_memory=False, usecols=cols_needed),
], ignore_index=True)

# Filter to New Jersey only
no2_nj = no2_raw[no2_raw['State Name'] == 'New Jersey'].copy()

print(f"Total NJ rows (all durations): {len(no2_nj)}")

# ── 2. FILTER TO 1-HOUR READINGS ──────────────────────────────────────────────
# NO2 data only comes in 1-hour samples. The EPA calculates the official NO2
# AQI from the daily 1-hour maximum, not a 24-hour average. We keep 1-hour
# readings and derive the daily max ourselves below.

no2_1hr = no2_nj[no2_nj['Sample Duration'] == '1 HOUR'].copy()

print(f"NJ rows after 1-hour filter: {len(no2_1hr)}")

# ── 3. PARSE DATES ────────────────────────────────────────────────────────────
no2_1hr['Date Local'] = pd.to_datetime(no2_1hr['Date Local'], format='mixed')
no2_1hr['Year'] = no2_1hr['Date Local'].dt.year

# ── 4. COLLAPSE TO ONE ROW PER STATION PER CALENDAR DAY ──────────────────────
# Each calendar day has up to 24 hourly rows per station. Before aggregating
# to the annual summary, we first reduce to one row per station per day.
#
# Per-day values:
#   daily_mean_reading  — average of all hourly readings that day
#   daily_max_aqi       — the highest AQI reading of the day (matches how the
#                         EPA officially reports NO2 AQI)
#   daily_1st_max       — the 1st Max Value field (EPA-computed daily max)

no2_daily = no2_1hr.groupby(
    ['Local Site Name', 'County Name', 'Latitude', 'Longitude', 'Year', 'Date Local']
).agg(
    daily_mean_reading = ('Arithmetic Mean', 'mean'),
    daily_max_aqi      = ('AQI', 'max'),       # official NO2 AQI uses daily 1-hr max
    daily_1st_max      = ('1st Max Value', 'max')
).reset_index()

print(f"Rows after daily collapse: {len(no2_daily)}")

# ── 5. SAVE INTERMEDIATE DAILY FILE ──────────────────────────────────────────
# This file is what you use for the NFL game-day comparison in Phase 2.
# It has one row per station per calendar day — exactly the right grain for
# filtering to specific match dates and building a control group.

no2_daily.to_csv('data/NO2/NO2_daily_clean_2023-2025.csv', index=False)
print("Saved: data/NO2/NO2_daily_clean_2023-2025.csv")

# ── 6. AGGREGATE TO ANNUAL SUMMARY ───────────────────────────────────────────
# Now reduce the daily file to one row per station per year.
#
#   avg_reading   — mean of daily averages (true annual daily-mean baseline)
#   avg_aqi       — mean of daily MAX AQI values (matches EPA reporting method)
#   peak_reading  — single worst day across the whole year
#   days_recorded — count of unique calendar days (nunique avoids double-count)

no2_summary = no2_daily.groupby(
    ['Local Site Name', 'County Name', 'Latitude', 'Longitude', 'Year']
).agg(
    avg_reading   = ('daily_mean_reading', 'mean'),
    avg_aqi       = ('daily_max_aqi', 'mean'),      # mean of daily maximums
    peak_reading  = ('daily_1st_max', 'max'),
    days_recorded = ('Date Local', 'nunique')
).reset_index()

# ── 7. SAVE ANNUAL SUMMARY ────────────────────────────────────────────────────
no2_summary.to_csv('data/NO2/NO2_summary_2023-2025.csv', index=False)
print("Saved: data/NO2/NO2_summary_2023-2025.csv")
print(no2_summary.to_string())

## June-July

## Days where AQI was bad

## NFL Game Day Data

In [ ]:
import pandas as pd

#game dates array
game_dates =  [
    "2023-09-11", "2023-09-24", "2023-10-01", "2023-10-15", "2023-11-06", 
    "2023-11-24", "2023-12-03", "2023-12-10", "2023-12-24", "2024-09-19", 
    "2024-09-29", "2024-10-14", "2024-10-31", "2024-11-17", "2024-12-01", 
    "2024-12-22", "2025-01-05", "2025-09-07", "2025-09-14", "2025-10-05", 
    "2025-10-19", "2025-11-09", "2025-11-30", "2025-12-07", "2025-12-28", 
    "2023-09-10", "2023-10-02", "2023-10-22", "2023-10-29", "2023-11-26", 
    "2023-12-11", "2023-12-31", "2024-01-07", "2024-09-08", "2024-09-26", 
    "2024-10-13", "2024-10-20", "2024-11-03", "2024-11-24", "2024-12-08", 
    "2024-12-15", "2024-12-28", "2025-09-21", "2025-09-28", "2025-10-09", 
    "2025-10-26", "2025-11-02", "2025-11-16", "2025-12-14", "2025-12-21" 
]

